In [1]:
from MACS3.IO.Parser import FragParser
from MACS3.IO.BedGraphIO import bedGraphIO
from MACS3.Utilities.Logger import logging
import numpy as np
from scipy import sparse
import anndata as ad
# we can use the pre-configured logging function from MACS3 to monitor memory usage...
logger = logging.getLogger("demo")
info = logger.info
# Note, you can replace `print` with `info` to show the time and memory usage at that moment.
from MACS3.Signal.PairedEndTrack import PETrackII
from MACS3.Signal.Region import Regions
import pandas as pd
import anndata as ad
from ncls import NCLS

In [2]:
# simulate a toy data

petrack = PETrackII()
petrack.add_loc(b"chr1", 0, 100, barcode=b"A", count=2)   # peak_1
petrack.add_loc(b"chr1", 70, 270, barcode=b"A", count=1)   # peak_1&2
petrack.add_loc(b"chr1", 0, 100, barcode=b"B", count=3)   # peak_1
petrack.add_loc(b"chr1", 175, 325, barcode=b"C", count=4)   # peak_2
petrack.finalize()

regions = Regions()
regions.add_loc(b"chr1", 0, 100)    # peak_1 A:3 B:3 C:0
regions.add_loc(b"chr1", 200, 300)  # peak_2 A:1 B:0 C:4
regions.add_loc(b"chr1", 500, 600)  # peak_3 A:0 B:0 C:0

In [3]:

def np_sort_search(petrack,regions):
        # barcode mapping
        barcode_items = sorted(petrack.barcode_dict.items(), key=lambda x: x[1])
        barcodes = [b.decode() if isinstance(b, (bytes, bytearray)) else str(b) for b, _ in barcode_items]
        barcode_id_to_row = {barcode_id: i for i, (_, barcode_id) in enumerate(barcode_items)}
        n_cells = len(barcodes)

        peak_names = []
        peak_data = []
        rows = []
        columns = []
        data = []

        regions.sort()
        peak_counter = 0

        for chrom in sorted(regions.regions.keys()):
            if chrom not in petrack.locations:
                continue

            regions_c = regions.regions[chrom]
            if not regions_c:
                continue

            local_peak_indices = []
            for (start, end) in regions_c:
                peak_counter += 1
                peak_names.append(f"peak_{peak_counter}")
                peak_data.append((chrom.decode() if isinstance(chrom, (bytes, bytearray)) else str(chrom), int(start), int(end)))
                local_peak_indices.append(peak_counter - 1)

            peak_starts = np.array([p[0] for p in regions_c], dtype=np.int64)
            peak_ends = np.array([p[1] for p in regions_c], dtype=np.int64)
            local_peak_indices = np.array(local_peak_indices, dtype=np.int64)

            fragment_locs = petrack.locations[chrom]
            fragment_barcodes = petrack.barcodes[chrom]
            if len(fragment_locs) == 0:
                continue

            for frag_idx, frag in enumerate(fragment_locs):
                frag_start = frag[0]
                frag_end = frag[1]
                bc_id = int(fragment_barcodes[frag_idx])
                row_id = barcode_id_to_row.get(bc_id, -1)
                if row_id < 0:
                    continue

                j = np.searchsorted(peak_starts, frag_start, side='left')
                while j < peak_starts.size and peak_starts[j] <= frag_end:
                    if peak_ends[j] > frag_start:
                        rows.append(row_id)
                        columns.append(int(local_peak_indices[j]))
                        data.append(int(frag[2]))
                    j += 1
        X = sparse.coo_matrix((data, (rows, columns)), shape=(n_cells, peak_counter), dtype=np.int32).tocsr()
        obs = pd.DataFrame(index=barcodes)
        var = pd.DataFrame(peak_data, columns=['chrom', 'start', 'end'], index=peak_names)
        adata_peaks = ad.AnnData(X, obs=obs, var=var)
        return adata_peaks


In [4]:
adata_np_ss = np_sort_search(petrack,regions)
print(adata_np_ss)
print(adata_np_ss.X.toarray())

AnnData object with n_obs × n_vars = 3 × 3
    var: 'chrom', 'start', 'end'
[[2 1 0]
 [3 0 0]
 [0 4 0]]


In [5]:
# ncls implementation
def build_anndata_ncls(petrack, regions, binary=False):
        regions.sort()

        # Barcode mapping
        barcode_items = sorted(petrack.barcode_dict.items(), key=lambda x: x[1])
        barcodes = [b.decode() if isinstance(b, (bytes, bytearray)) else str(b) for b, _ in barcode_items]
        barcode_ids = np.array([i for _, i in barcode_items], dtype=np.int64)

        n_cells = len(barcodes)
        if n_cells == 0:
            return ad.AnnData(X=sparse.csr_matrix((0, 0), dtype=np.int32))

        id_map = np.full(barcode_ids.max() + 1, -1, dtype=np.int64)
        id_map[barcode_ids] = np.arange(n_cells, dtype=np.int64)

        # Outputs
        peak_names = []
        peak_data = []
        rows = []
        cols = []
        data = []
        peak_counter = 0

        for chrom in sorted(regions.regions.keys()):
            if chrom not in petrack.locations:
                continue

            regions_c = regions.regions[chrom]
            if not regions_c:
                continue

            # register peaks (global)
            local_peak_indices = []
            chrom_str = chrom.decode() if isinstance(chrom, (bytes, bytearray)) else str(chrom)
            for (start, end) in regions_c:
                peak_counter += 1
                peak_names.append(f"peak_{peak_counter}")
                peak_data.append((chrom_str, int(start), int(end)))
                local_peak_indices.append(peak_counter - 1)

            # fragments on this chromosome
            fragment_locs = petrack.locations[chrom][:petrack.size[chrom]]
            fragment_barcodes = petrack.barcodes[chrom][:petrack.size[chrom]]
            if len(fragment_locs) == 0:
                continue

            frag_starts = fragment_locs['l'].astype(np.int64)
            frag_ends = fragment_locs['r'].astype(np.int64)
            frag_counts = fragment_locs['c'].astype(np.int64)
            frag_bc_ids = fragment_barcodes.astype(np.int64)

            frag_rows = id_map[frag_bc_ids]  # barcode -> row index

            idx = np.arange(len(frag_starts), dtype=np.int64)
            ncls = NCLS(frag_starts, frag_ends, idx)

            # Query overlaps: count any fragment that overlaps the peak
            for peak_idx, (peak_start, peak_end) in enumerate(regions_c):
                for frag_start, frag_end, frag_i in ncls.find_overlap(peak_start, peak_end):
                    row_id = frag_rows[frag_i]
                    if row_id >= 0:
                        rows.append(row_id)
                        cols.append(local_peak_indices[peak_idx])
                        data.append(int(frag_counts[frag_i]))
                            
        X = sparse.coo_matrix((data, (rows, cols)),
                            shape=(n_cells, peak_counter),
                            dtype=np.int32).tocsr()

        obs = pd.DataFrame(index=barcodes)
        var = pd.DataFrame(peak_data, columns=["chrom", "start", "end"], index=peak_names)
        return ad.AnnData(X=X, obs=obs, var=var)


In [6]:
adata_ncls = build_anndata_ncls(petrack,regions)
print(adata_ncls)
print(adata_ncls.X.toarray())

AnnData object with n_obs × n_vars = 3 × 3
    var: 'chrom', 'start', 'end'
[[3 1 0]
 [3 0 0]
 [0 4 0]]


In [7]:
# two pointer sweep like exclude

from operator import itemgetter

def two_pointer_sweep(petrack, regions):

    peak_names = []
    peak_data = []

    rows = [] #barcode index
    columns = [] #peaks index
    data = [] #count data

    barcode_items = sorted(petrack.barcode_dict.items(), key=itemgetter(1))
    barcodes = [b.decode() if isinstance(b, (bytes, bytearray)) else str(b) for b, _ in barcode_items]
    n_cells = len(barcodes)
    if n_cells:
        barcode_ids = np.fromiter((barcode_id for _, barcode_id in barcode_items),
                                    dtype=np.int64,
                                    count=n_cells)
        barcode_id_to_row = np.full(int(barcode_ids[-1]) + 1, -1, dtype=np.int64)
        barcode_id_to_row[barcode_ids] = np.arange(n_cells, dtype=np.int64)
    else:
        barcode_id_to_row = np.zeros(0, dtype=np.int64)

    regions.sort()
    peak_counter = 0

    for chrom in regions.regions.keys():
        if chrom not in petrack.locations:
            continue

        regions_c = regions.regions[chrom]
        if not regions_c:
            continue

        local_peak = []
        peak_starts = []
        peak_ends = []
        chrom_str = chrom.decode() if isinstance(chrom, (bytes, bytearray)) else str(chrom)
        ### regions empty skip
        for (start, end) in regions_c:
            peak_counter += 1
            peak_names.append(f"peak_{peak_counter}")
            peak_data.append((chrom_str, start, end))
            local_peak.append(peak_counter - 1)
            peak_starts.append(start)
            peak_ends.append(end)

        fragment_locs = petrack.locations[chrom]

        if len(fragment_locs) == 0:
            continue

        fragment_barcodes = petrack.barcodes[chrom]
        frag_starts = fragment_locs['l']
        frag_ends = fragment_locs['r']
        frag_counts = fragment_locs['c']
        frag_rows = barcode_id_to_row[fragment_barcodes]
        n_frags = len(fragment_locs)

        frag_idx = 0
        local_peak_idx = 0
        frag_start = frag_starts[frag_idx]
        frag_end = frag_ends[frag_idx]
        peak_start = peak_starts[local_peak_idx]
        peak_end = peak_ends[local_peak_idx]
        remaining_frag_len = n_frags - 1
        remaining_peak_len = len(regions_c) - 1
        back_trace_frag = 0

        while True:
            if frag_start <= peak_end and peak_start <= frag_end:
                row_id = frag_rows[frag_idx]
                if row_id >= 0:
                    rows.append(int(row_id))
                    columns.append(local_peak[local_peak_idx])
                    data.append(int(frag_counts[frag_idx]))

                if frag_end > peak_end:
                    back_trace_frag += 1

                if remaining_frag_len:
                    remaining_frag_len -= 1
                    frag_idx += 1
                    frag_start = frag_starts[frag_idx]
                    frag_end = frag_ends[frag_idx]
                    continue
                else:
                    break

            if frag_end < peak_end:
                if remaining_frag_len:
                    remaining_frag_len -= 1
                    frag_idx += 1
                    frag_start = frag_starts[frag_idx]
                    frag_end = frag_ends[frag_idx]
                else:
                    break
            else:
                if remaining_peak_len:
                    remaining_peak_len -= 1
                    local_peak_idx += 1
                    peak_start = peak_starts[local_peak_idx]
                    peak_end = peak_ends[local_peak_idx]
                    if back_trace_frag:
                        frag_idx -= back_trace_frag
                        remaining_frag_len += back_trace_frag
                        frag_start = frag_starts[frag_idx]
                        frag_end = frag_ends[frag_idx]
                        back_trace_frag = 0
                else:
                    break

    n_peaks = peak_counter
    obs = pd.DataFrame(index=barcodes)
    var = pd.DataFrame(peak_data, columns=['chrom', 'start', 'end'], index=peak_names)

    x = sparse.csr_matrix((data,(rows,columns)),shape=(n_cells,n_peaks))
    adata_peaks_loop = ad.AnnData(X=x, obs=obs, var=var)
    return adata_peaks_loop


In [8]:
adata_two = two_pointer_sweep(petrack,regions)
print(adata_two)
print(adata_two.X.toarray())

AnnData object with n_obs × n_vars = 3 × 3
    var: 'chrom', 'start', 'end'
[[3 1 0]
 [3 0 0]
 [0 4 0]]


In [9]:
## test andata from package

a = petrack.return_anndata(regions, method = "ncls")
a.X.toarray()


INFO  @ 13 Mar 2026 13:31:57: [150 MB] Collecting data... 
INFO  @ 13 Mar 2026 13:31:57: [150 MB] Done with data collection 
INFO  @ 13 Mar 2026 13:31:57: [150 MB] Done with AnnData construction 


using ncls


array([[3, 1, 0],
       [3, 0, 0],
       [0, 4, 0]], dtype=int32)